In [ ]:
! pip install pingouin

# Load packages
import pandas as pd
import numpy as np

import re

import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import shapiro, ttest_rel
from pingouin import intraclass_corr, corr, cronbach_alpha, wilcoxon

Generate synthetic dataset to code data analysis notebook

In [ ]:
np.random.seed(100)  # reproducible fake data
n = 10

phq_options = ["Not at all", "Several days", "More than half the days", "Nearly every day"]
likert_5_labels = ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"]

phq_cols = [
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.1. Little interest or pleasure in doing things",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.2. Feeling down, depressed or hopeless",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.3. Trouble falling or staying asleep, or sleeping too much",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.4. Feeling tired or having little energy",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.5. Poor appetite or overeating",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.6. Feeling bad about yourself - or that you are a failure or have let yourself or your family down",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.7. Trouble concentrating on things, such as reading the newspaper or watching television",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.8. Moving or speaking so slowly that other people could have noticed? Or the opposite - being so fidgety or re",
    "Over the last 2 weeks, how often have you been bothered by any of the following problems?.9. Thoughts that you would be better off dead or of hurting yourself in some way",
]

# item 9 forced low/zero to avoid generating fake self-harm-positive rows
def fake_phq_row():
    row = list(np.random.choice(phq_options, size=8))
    row.append(np.random.choice(["Not at all", "Not at all", "Not at all", "Several days"]))  # item 9 skewed to 0
    return row

likert_section_cols = [
    "Section C. Conversation Experience with HopeBot.The chatbot was easy to use.",
    "Section C. Conversation Experience with HopeBot.The conversation felt clear and easy to follow.",
    "Section C. Conversation Experience with HopeBot.The chatbot felt supportive.",
    "Section C. Conversation Experience with HopeBot.I did not feel judged by the chatbot.",
    "Section C. Conversation Experience with HopeBot.The chatbot handled sensitive topics appropriately.",
    "Section C. Conversation Experience with HopeBot.I felt comfortable completing the conversation with the chatbot.",
    "Section D. Questionnaire Process.The transition from free conversation to questionnaire felt natural.",
    "Section D. Questionnaire Process.The questionnaire itself was easy to complete in this format.",
    "Section D. Questionnaire Process.The chatbot understood my responses accurately.",
    "Section E. Trust and perceived quality.I trusted the chatbot's screening process.",
    "Section E. Trust and perceived quality.I trusted the results the chatbot produced.",
    "Section E. Trust and perceived quality.I understood what my screening outcome meant.",
    "Section E. Trust and perceived quality.I felt the chatbot understood the main issues I was facing.",
    "Section E. Trust and perceived quality.The chatbot's recommendation matched what I was experiencing.",
    "Section E. Trust and perceived quality.The chatbot's explanation of my result was hard to follow.",
    "Section E. Trust and perceived quality..I would consider the chatbot a useful first step before seeking further help.",
    "Section F. Agent Functionality.The mental health resources provided were relevant to me.",
    "Section F. Agent Functionality.The psycho-education material was trustworthy.",
    "Section F. Agent Functionality.The psycho-education material was relevant to my situation.",
    "Section F. Agent Functionality.The explanation of my PHQ-9 score was clear.",
    "Section F. Agent Functionality.The summary was useful as a reference I could return to.",
    "Section F. Agent Functionality.I could see myself using the self-care tips recommended.",
    "Section F. Agent Functionality.I would be comfortable using HopeBot as a first point of contact that helps connect me to mental health care",
    "Section F. Agent Functionality.I would be open to using a scheduling feature in the future.",
    "Section F. Agent Functionality.I would be open to using a session preparation feature in the future.",
]

free_text_cols = [
    "What did you find most helpful about HopeBot - in the screening, the support that followed, or both?",
    "What did you find confusing, frustrating, or unhelpful - in the screening, the support that followed, or the switch between them?",
    "Would you use HopeBot again as a first step for mental health screening and the care coordination support that followed? Why or why not?",
    "If you were struggling between scheduled appointments, how comfortable would you feel turning to HopeBot for support in that moment? Why or why not?",
    "Was there anything that made you hesitant to trust or rely on HopeBot? If so, please explain.",
    "How did interacting with HopeBot make you feel?",
]

data = {
    "Id": range(1, n + 1),
    "Start time": pd.date_range("2026-06-01 09:00", periods=n, freq="3h"),
    "Completion time": pd.date_range("2026-06-01 09:15", periods=n, freq="3h"),
    "Email": [f"participant{i}@example.com" for i in range(1, n + 1)],
    "Name": [f"Test Participant {i}" for i in range(1, n + 1)],
    "Please enter the Participant ID provided to you by HopeBot at the end of your conversation (e.g. S01):": [f"S{i:02d}" for i in range(1, n + 1)],
    "Age Group": np.random.choice(["18-24", "25-34", "35-44", "45-54", "55+"], n),
    "Gender": np.random.choice(["Male", "Female", "Non-binary", "Prefer not to say"], n),
    "Ethnicity": np.random.choice(["White", "Asian", "Black", "Mixed", "Other"], n),
    "Education Level": np.random.choice(["Undergraduate", "Postgraduate", "A-levels/equivalent"], n),
    "Employment Status": np.random.choice(["Employed full-time", "Student", "Employed part-time", "Unemployed"], n),
    "Have you ever used mental health services before (e.g., therapy, counselling, psychiatric care, helplines)?": np.random.choice(["Yes", "No"], n),
    "Before this study, how familiar were you with large language models (LLMs) such as ChatGPT?": np.random.choice(likert_5, n),
    "Before this session, have you ever used an AI chatbot for mental health or mental health support?": np.random.choice(["Yes", "No"], n),
}

# add PHQ-9 items row by row
phq_data = [fake_phq_row() for _ in range(n)]
for col_idx, col_name in enumerate(phq_cols):
    data[col_name] = [row[col_idx] for row in phq_data]

data["What was your HopeBot-assisted PHQ-9 score?"] = np.random.randint(0, 27, n)

for col in likert_section_cols:
    data[col] = np.random.choice(likert_5_labels, n)

for col in free_text_cols:
    data[col] = [f"Sample free-text response {i} for testing." for i in range(1, n + 1)]

df = pd.DataFrame(data)
df.columns = df.columns.str.strip()
# df.to_csv("synthetic_phq9_data.csv", index=False)
df.head(3)

In [ ]:
df = df.iloc[:, 5:]
# rename columns for readibility
df = df.rename(columns={"What was your HopeBot-assisted PHQ-9 score?": "hopebot_phq9_score",
                        'Please enter the Participant ID provided to you by HopeBot at the end of your conversation (e.g. S01):': 'participant_id'})

In [ ]:
# check for missing values
print(df.isna().any().any())
print(df.isnull().any().any())

## PHQ-9 Scores Analysis

Code the self-administered PHQ-9 responses

In [ ]:
# grab all columns matching the PHQ-9 stem
phq_cols = [c for c in df.columns if "Over the last 2 weeks, how often have you been bothered by any of the following problems?" in c]

# sort by the leading number so q1...q9 map correctly
phq_cols_sorted = sorted(phq_cols, key=lambda c: int(re.search(r'\.(\d+)\.', c).group(1)))

rename_map = {col: f"q{i+1}" for i, col in enumerate(phq_cols_sorted)}
df = df.rename(columns=rename_map)

In [ ]:
mapping = {'Not at all': 0, 'Several days': 1, 'More than half the days': 2, 'Nearly every day': 3}

new_phq_cols = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9']
df[new_phq_cols] = df[new_phq_cols].apply(lambda col: col.map(mapping))
df['self_phq9_score'] = df[new_phq_cols].sum(axis = 1)

Shapiro-Wilk Test for Normality

In [ ]:
df['phq9_diff'] = df['hopebot_phq9_score'] - df['self_phq9_score']

W, p = shapiro(df['phq9_diff'])
print(f'Shaprio-Wilk normality assessment gave a W of {W:.3f} (p-value = {p:.3f})')

Wilcoxon Signed-Rank Test & Paired-Samples T-test

In [ ]:
if p > 0.05:
    print('p > 0.05. Hence, we fail to reject the null hypothesis and there is no significant evidence against normality.' \
    'Differences are consistent with a normal distribution → proceed with both wilcoxon signed-rank t-test and paired t-test.')

    # Wilcoxon Signed-Rank test
    print('\nWilcoxon Signed-Rank Test:')
    result = wilcoxon(df['hopebot_phq9_score'], df['self_phq9_score'])
    W = result['W_val'].values[0]
    p = result['p_val'].values[0]
    r = result['RBC'].values[0]
    print(f"Wilcoxon signed-rank: W = {W}, p = {p:.3f}, r = {r:.3f}")

    # Paired-samples t-test
    print('\nPaired-samples Test:')
    ttest_result = ttest_rel(df['hopebot_phq9_score'], df['self_phq9_score'])
    print(f'Paired-samples T-test statistic: {ttest_result.statistic:.3f}, p = {ttest_result.pvalue:.3f}')
else:
    print('p ≤ 0.05. Hence, we reject the null hypothesis and differences significantly deviate from normality. → use the Wilcoxon signed-rank test instead')

    # Wilcoxon Signed-Rank test
    print('\nWilcoxon Signed-Rank Test:')
    result = wilcoxon(df['hopebot_phq9_score'], df['self_phq9_score'])
    W = result['W_val'].values[0]
    p = result['p_val'].values[0]
    r = result['RBC'].values[0]
    print(f"Wilcoxon signed-rank: W = {W}, p = {p:.3f}, r = {r:.3f}")

Calculate correlation (Spearman's correlation + Intraclass Correlation)

In [ ]:
# Spearman's Correlation
print("Spearman's rank correlation:")
print(corr(df['hopebot_phq9_score'], df['self_phq9_score'], method='spearman'))

# Convert df to long format
cols = ['participant_id', 'hopebot_phq9_score', 'self_phq9_score']
long_df = pd.melt(df[cols], id_vars='participant_id', value_vars=['hopebot_phq9_score', 'self_phq9_score']).sort_values(by = 'participant_id')
long_df.rename(columns = {'participant_id':'participant_id',
                        'variable': 'admin_mode',
                        'value': 'phq9_score'}, inplace=True)

# Intra-class correlation
print('\nIntraclass Correlation:')
icc_results = intraclass_corr(data = long_df,
                targets = 'participant_id',
                raters = 'admin_mode',
                ratings = 'phq9_score')
print(icc_results.set_index('Type'))

Severity-band concordance

In [ ]:
def severity_band(score):
    if score <= 4: return "Minimal"
    elif score <= 9: return "Mild"
    elif score <= 14: return "Moderate"
    elif score <= 19: return "Moderately severe"
    else: return "Severe"

df['self_band'] = df['self_phq9_score'].apply(severity_band)
df['hopebot_band'] = df['hopebot_phq9_score'].apply(severity_band)

df['band_match'] = df['self_band'] == df['hopebot_band']

In [ ]:
print(f'Percentage of participants with severity band concordance: {df['band_match'].sum()/len(df) * 100:.1f}%')

In [ ]:
# Check for mismatched severity
print(df[~df['band_match']][['participant_id', 'self_band', 'hopebot_band']])

Bland-Altman Plot

In [ ]:
#create Bland-Altman plot                  
f, ax = plt.subplots(1, figsize = (8,5))
sm.graphics.mean_diff_plot(df.hopebot_phq9_score, df.self_phq9_score, ax = ax)

#display Bland-Altman plot
plt.show()

## Survey Analysis (Likert-Scale)

In [ ]:
likert_df = df[likert_section_cols]
likert_df.head(3)

In [ ]:
# Code likert scale responses
likert_mapping = {"Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5}
likert_df = likert_df.apply(lambda col: col.map(likert_mapping))

In [ ]:
# Reverse coding
likert_df["Section E. Trust and perceived quality.The chatbot's explanation of my result was hard to follow."] = 6 - likert_df["Section E. Trust and perceived quality.The chatbot's explanation of my result was hard to follow."]

Per-item level

In [ ]:
# Survey items by section mapping
section_c = [
    "Section C. Conversation Experience with HopeBot.The chatbot was easy to use.",
    "Section C. Conversation Experience with HopeBot.The conversation felt clear and easy to follow.",
    "Section C. Conversation Experience with HopeBot.The chatbot felt supportive.",
    "Section C. Conversation Experience with HopeBot.I did not feel judged by the chatbot.",
    "Section C. Conversation Experience with HopeBot.The chatbot handled sensitive topics appropriately.",
    "Section C. Conversation Experience with HopeBot.I felt comfortable completing the conversation with the chatbot.",
]
section_d = [
    "Section D. Questionnaire Process.The transition from free conversation to questionnaire felt natural.",
    "Section D. Questionnaire Process.The questionnaire itself was easy to complete in this format.",
    "Section D. Questionnaire Process.The chatbot understood my responses accurately.",
]
section_e = [
    "Section E. Trust and perceived quality.I trusted the chatbot's screening process.",
    "Section E. Trust and perceived quality.I trusted the results the chatbot produced.",
    "Section E. Trust and perceived quality.I understood what my screening outcome meant.",
    "Section E. Trust and perceived quality.I felt the chatbot understood the main issues I was facing.",
    "Section E. Trust and perceived quality.The chatbot's recommendation matched what I was experiencing.",
    "Section E. Trust and perceived quality.The chatbot's explanation of my result was hard to follow.",
    "Section E. Trust and perceived quality..I would consider the chatbot a useful first step before seeking further help.",
]
section_f = [
    "Section F. Agent Functionality.The mental health resources provided were relevant to me.",
    "Section F. Agent Functionality.The psycho-education material was trustworthy.",
    "Section F. Agent Functionality.The psycho-education material was relevant to my situation.",
    "Section F. Agent Functionality.The explanation of my PHQ-9 score was clear.",
    "Section F. Agent Functionality.The summary was useful as a reference I could return to.",
    "Section F. Agent Functionality.I could see myself using the self-care tips recommended.",
    "Section F. Agent Functionality.I would be comfortable using HopeBot as a first point of contact that helps connect me to mental health care",
    "Section F. Agent Functionality.I would be open to using a scheduling feature in the future.",
    "Section F. Agent Functionality.I would be open to using a session preparation feature in the future.",
]

In [ ]:
# Per-item descriptive statistics
# section C
print('Section C:')
display(likert_df[section_c].describe())

# section d
print('\nSection D:')
display(likert_df[section_d].describe())

# section e
print('\nSection E:')
display(likert_df[section_e].describe())

# section f
print('\nSection F:')
display(likert_df[section_f].describe())

Per-section level

In [ ]:
print('Section C:')
df['section_c_composite'] = likert_df[section_c].mean(axis=1)
print(f'Median: {df['section_c_composite'].median():.3f}')
print(f'IQR:\n{df['section_c_composite'].quantile([0.25, 0.75])}\n')

print('Section D:')
df['section_d_composite'] = likert_df[section_d].mean(axis=1)
print(f'Median: {df['section_d_composite'].median():.3f}')
print(f'IQR:\n{df['section_d_composite'].quantile([0.25, 0.75])}\n')

print('Section E:')
df['section_e_composite'] = likert_df[section_e].mean(axis=1)
print(f'Median: {df['section_e_composite'].median():.3f}')
print(f'IQR:\n{df['section_e_composite'].quantile([0.25, 0.75])}\n')

print('Section F:')
df['section_f_composite'] = likert_df[section_f].mean(axis=1)
print(f'Median: {df['section_f_composite'].median():.3f}')
print(f'IQR:\n{df['section_f_composite'].quantile([0.25, 0.75])}\n')

In [ ]:
# section c
alpha_c, ci_c = cronbach_alpha(data=likert_df[section_c])
print("Section C:")
print(f"Cronbach's Alpha: {alpha_c:.3f}, CI {ci_c}\n")

# section d
alpha_d, ci_d = cronbach_alpha(data=likert_df[section_d])
print("Section D:")
print(f"Cronbach's Alpha: {alpha_d:.3f}, CI {ci_d}\n")

# section e
alpha_e, ci_e = cronbach_alpha(data=likert_df[section_e])
print("Section E:")
print(f"Cronbach's Alpha: {alpha_e:.3f}, CI {ci_e}\n")

# section f
alpha_f, ci_f = cronbach_alpha(data=likert_df[section_f])
print("Section F:")
print(f"Cronbach's Alpha: {alpha_f:.3f}, CI {ci_f}\n")

Diverged Bar Chart

In [ ]:
def diverging_stacked_bar(likert_df, item_cols, item_labels=None, title="Likert Response Distribution"):
    """
    likert_df: dataframe with raw (uncoded) Likert text responses
    item_cols: list of column names to include
    item_labels: optional shorter display labels for each item (same order as item_cols)
    """
    categories = ["Strongly disagree", "Disagree", "Neutral", "Agree", "Strongly agree"]
    colors = ["#c0392b", "#e74c3c", "#bdc3c7", "#2ecc71", "#27ae60"]  # red -> grey -> green

    # % of respondents per category, per item
    pct = pd.DataFrame({
        col: likert_df[col].value_counts(normalize=True).reindex(categories, fill_value=0) * 100
        for col in item_cols
    }).T

    labels = item_labels if item_labels else item_cols

    # Diverging: negative categories go left of zero, positive go right
    neg = pct[["Strongly disagree", "Disagree"]].copy()
    neg["Strongly disagree"] *= -1
    neg["Disagree"] *= -1
    neutral_half = pct["Neutral"] / 2
    pos = pct[["Agree", "Strongly agree"]]

    fig, ax = plt.subplots(figsize=(9, 0.5 * len(item_cols) + 1))

    left = -neutral_half.values.copy()
    ax.barh(labels, neg["Strongly disagree"], left=left, color=colors[0], label="Strongly disagree")
    left += neg["Strongly disagree"].values
    ax.barh(labels, neg["Disagree"], left=left, color=colors[1], label="Disagree")

    left = -neutral_half.values.copy()
    ax.barh(labels, pct["Neutral"], left=left, color=colors[2], label="Neutral")

    left = neutral_half.values.copy()
    ax.barh(labels, pos["Agree"], left=left, color=colors[3], label="Agree")
    left += pos["Agree"].values
    ax.barh(labels, pos["Strongly agree"], left=left, color=colors[4], label="Strongly agree")

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("% of respondents")
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    return fig, ax

In [ ]:
# Section C
fig, ax = diverging_stacked_bar(df, section_c, title="Section C: Conversation Experience")
# plt.savefig("section_c_diverging.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Section D
fig, ax = diverging_stacked_bar(df, section_d, title="Section D: Questionnaire Process")
# plt.savefig("section_d_diverging.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Section E
fig, ax = diverging_stacked_bar(df, section_e, title="Section E: Trust and Perceived Quality")
# plt.savefig("section_d_diverging.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Section F
fig, ax = diverging_stacked_bar(df, section_f, title="Section F: Agent Functionality")
# plt.savefig("section_d_diverging.png", dpi=300, bbox_inches="tight")
plt.show()

## Thematic Analysis

In [ ]:
open_ended = ["participant_id",
            "What did you find most helpful about HopeBot - in the screening, the support that followed, or both?",
            "What did you find confusing, frustrating, or unhelpful - in the screening, the support that followed, or the switch between them?",
            "Would you use HopeBot again as a first step for mental health screening and the care coordination support that followed? Why or why not?",
            "If you were struggling between scheduled appointments, how comfortable would you feel turning to HopeBot for support in that moment? Why or why not?",
            "Was there anything that made you hesitant to trust or rely on HopeBot? If so, please explain.",
            "How did interacting with HopeBot make you feel?"]
open_df = df[open_ended]

In [ ]:
question_map = {
    "Q1_helpful": "What did you find most helpful about HopeBot - in the screening, the support that followed, or both?",
    "Q2_confusing": "What did you find confusing, frustrating, or unhelpful - in the screening, the support that followed, or the switch between them?",
    "Q3_reuse": "Would you use HopeBot again as a first step for mental health screening and the care coordination support that followed? Why or why not?",
    "Q4_between_appts": "If you were struggling between scheduled appointments, how comfortable would you feel turning to HopeBot for support in that moment? Why or why not?",
    "Q5_trust": "Was there anything that made you hesitant to trust or rely on HopeBot? If so, please explain.",
    "Q6_feelings": "How did interacting with HopeBot make you feel?"
}

# rename using the short codes as keys
open_df = open_df.rename(columns={v: k for k, v in question_map.items()})

open_df_long = open_df.melt(
    id_vars=["participant_id"],
    value_vars=list(question_map.keys()),
    var_name="question_short",
    value_name="response_text"
)

open_df_long["question_full_text"] = open_df_long["question_short"].map(question_map)
# export df for coding outside in excel
open_df_long.to_csv("responses_for_coding.csv", index=False)

In [ ]:
# reload csv file
# codes = pd.read_csv("responses_for_coding.csv")

In [ ]:
# Check coding coverage (sanity check)
coded_pairs = set(zip(codes.participant_id, codes.question_short))
all_pairs = set(zip(open_df_long.participant_id, open_df_long.question_short.map(
    {v: k for k, v in question_map.items()}
)))

uncoded = all_pairs - coded_pairs

In [ ]:
# Check for cross-question patterns
pivot = codes.pivot_table(
    index = 'participant_id', columns = 'question_short',
    values = 'code_label', aggfunc = lambda x: ";".join(x)
) # check for same underlying idea, different code label, different question

In [ ]:
# TODO: candidate_theme mapping populated from real coded data in local analysis notebook (not version-controlled per ethics requirements — see Appendix X / data management section)
# codes['candidate_theme'] = codes["code_label"].map({...})

# TODO: check which questions feed into each theme — cross-question themes are often stronger
# codes.groupby("candidate_theme")["question_short"].unique()

In [ ]:
# TODO: pull extracts for report write-up
# e.g. codes[codes.candidate_theme == "Authenticity concerns"][["participant_id", "question_short", "data_extract"]]